# Proteger las neuronas importantes: ¿el arreglo WSI mejora si no toca lo que el razonamiento normal necesita?

El arreglo WSI v3 (capas 11, 15, 7; alpha 6; 64 neuronas por capa) elige neuronas solo por cuánto empujan hacia el loop. Nunca mide **cuánto daño hace editarlas cuando el modelo razona bien**.

Aquí se mide ese daño en las respuestas buenas del propio modelo (correctas, sin loop, de GSM8K *train*): por cada neurona, cuánto sube la pérdida si se edita (un backward pass, fórmula exacta de primer orden para esta edición). Con eso se arman 3 variantes protegidas:

- **skip10 / skip25:** salta las neuronas que están en el 10% / 25% más dañino de su capa y toma las siguientes hasta 64.
- **drop10:** quita el 10% más dañino, sin reponer.

Se elige en los mismos 64 problemas de dev a 16k que eligieron v3. Si alguna gana, va al test (200 problemas, siempre activo y vía MoI) contra v3 (40,5% / 55%).

**Necesita** la corrida v3 en tu Drive (`wsi_runs/discover_loop_fix`). **Uso:** `L4` → `Run all` → acepta Drive. ~1 h. El resultado queda en `MyDrive/wsi_runs/FINALES/protect_neurons.json` y el runtime se apaga solo si ese archivo existe.

In [ ]:
W = "/content/work"
REPO = "/content/wsi"
S = f"{REPO}/scripts/protect_neurons.py"
RUN_NAME = "protect_neurons"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm
import vllm, transformers; print("vllm", vllm.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, glob, shutil, json, subprocess
from google.colab import drive
drive.mount("/content/drive")
V3 = "/content/drive/MyDrive/wsi_runs/discover_loop_fix"          # la corrida v3 (solo se lee)
DRIVE = f"/content/drive/MyDrive/wsi_runs/{RUN_NAME}"
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
os.makedirs(DRIVE, exist_ok=True); os.makedirs(FINALES, exist_ok=True); os.makedirs(W, exist_ok=True)
NECESITO = ["config.json", "requests_elicit.jsonl", "outputs_elicit.jsonl", "wsi_directions.npz", "diagnosis.json",
            "requests_dev16.jsonl", "outputs_dev16_base.jsonl", "outputs_dev16_fa3_a6.jsonl", "requests_test.jsonl",
            "outputs_test_base.jsonl", "requests_test_moi.jsonl", "outputs_test_wsi3.jsonl", "outputs_test_moi3.jsonl"]
falta = [f for f in NECESITO if not os.path.exists(f"{V3}/{f}")]
assert not falta, f"no encuentro la corrida v3 en Drive: faltan {falta}"
for f in NECESITO:
    shutil.copy2(f"{V3}/{f}", W)
for f in glob.glob(f"{DRIVE}/*"):          # lo que esta corrida ya terminó
    shutil.copy2(f, W)

def guardar():
    for pat in ("*.json", "*.jsonl", "*.npz"):
        for f in glob.glob(f"{W}/{pat}"):
            if os.path.basename(f) not in NECESITO or os.path.basename(f) == "config.json":
                shutil.copy2(f, DRIVE)

def hecho(*archivos):
    return all(os.path.exists(f"{DRIVE}/{a}") for a in archivos)

print("v3 recuperada; ya hecho aquí:", sorted(os.listdir(DRIVE)) or "nada")

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --filter=blob:none --sparse https://github.com/krexito/weight-space-index.git {REPO}
    !cd {REPO} && git sparse-checkout set scripts
else:
    !cd {REPO} && git pull -q
assert os.path.exists(S), f"{S} no está en GitHub todavía: falta hacer push del repo"
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
cfg = json.load(open(f"{W}/config.json"))
if not os.path.exists(cfg["base_dir"]):
    from huggingface_hub import snapshot_download
    snapshot_download("Qwen/Qwen3.5-0.8B", local_dir=cfg["base_dir"])
print("commit", COMMIT, "| modelo base en", cfg["base_dir"])

In [ ]:
# 1. mapa de daño: qué neuronas necesita el razonamiento bueno (backward en respuestas correctas sin loop)
if not hecho("protect.json"):
    !python {S} harm --work {W}
    guardar()
p = json.load(open(f"{W}/protect.json"))
print("neuronas por variante:", p["n_columns"])
print("NLL en respuestas buenas apartadas (menos = menos daño):", p["heldout_nll"])
print("v3 reproducido:", p["orig_matches_v3_columns"])

In [ ]:
# 2. las 3 variantes protegidas en los 64 problemas dev a 16k (v3 original ya está medido)
for v in ("skip10", "skip25", "drop10"):
    out = f"outputs_dev16_protect_{v}.jsonl"
    if hecho(out):
        continue
    CK = f"{W}/protect-{v}"
    !python {S} build --work {W} --variant {v} --out-dir {CK}
    !python {S} gen --work {W} --model {CK} --requests requests_dev16.jsonl --out {out}
    guardar()
    shutil.rmtree(CK, ignore_errors=True)
!python {S} pick --work {W}
guardar()
CH = json.load(open(f"{W}/protect.json"))["choice"]
print("elegido:", CH)

In [ ]:
# 3. test del ganador (si ganó el original, no hay nada nuevo que probar)
if CH != "orig":
    CK = f"{W}/protect-{CH}"
    if not hecho("outputs_test_protect.jsonl", "outputs_test_moi_protect.jsonl"):
        !python {S} build --work {W} --variant {CH} --out-dir {CK}
    if not hecho("outputs_test_protect.jsonl"):
        !python {S} gen --work {W} --model {CK} --requests requests_test.jsonl --out outputs_test_protect.jsonl
        guardar()
    if not hecho("outputs_test_moi_protect.jsonl"):
        !python {S} gen --work {W} --model {CK} --requests requests_test_moi.jsonl --out outputs_test_moi_protect.jsonl
        guardar()
!python {S} report --work {W}
guardar()

In [ ]:
# 4. resultado final a Drive (FINALES): primero a un tmp, luego os.replace
import datetime
res = json.load(open(f"{W}/results_protect_neurons.json"))
res["_final"] = {"complete": True, "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
                 "notebook": "protect_neurons_colab.ipynb", "commit": COMMIT}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as f:
    json.dump(res, f, indent=1)
os.replace(dst + ".tmp", dst)
print("guardado en", dst)
try:
    from google.colab import files
    files.download(dst)
except Exception as e:
    print("descarga directa no disponible:", e)

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(90)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")